# WetlandMapper: from an outline to wetland maps, statistics and GeoTIFFs

Wetlands Indaba 2026, 90-minute workshop. Runs in Google Colab or on your own computer. You do not need to write code.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manudeo/Wetlands_Indaba_2026_workshop/blob/main/notebooks/WetlandMapper_workshop.ipynb)

Everything here is done with [WetlandMapper](https://github.com/manudeo/wetlandmapper) itself: it downloads the satellite record from Google Earth Engine (`fetch`), classifies **wetland dynamics** (`classify_dynamics`) and **Wetland Cover Types** (`classify_wct_ema`), and summarises and plots the results (`summarize_*`, `class_area_timeseries`, `plot_*`). The notebook adds only a few lines for reading your outline, caching downloads and writing files.

| Part | What | Minutes |
|---|---|---|
| Setup | Run the first cell; optional Earth Engine sign-in | 10 |
| **A** | **Find the wetlands** in an area: the Agulhas Plain, compared with the national wetland map | 10 |
| **B** | **One wetland through 36 years**: a pan on the Agulhas Plain; dynamics, cover types, rainfall | 20 |
| **C** | **Cover types this week**: the Langebaan Ramsar site, to compare with your field visit | 10 |
| **Your area** | In each part, switch *AREA* to *my own outline* and run it again | 20 |
| Wrap-up | Reading results, limits, what else WetlandMapper can do | 10 |

**How to use it.** Press the play button on each cell, in order. Cells with settings show a form on the right; click *Show code* to see what runs. The three examples use sample data that were downloaded in advance and cached in the repository: no account needed. **Your own area and the live "this week" map need Google Earth Engine.**

## Setup

In [ ]:
#@title 0. Setup: run this first (about a minute) { display-mode: "form" }
USE_EARTH_ENGINE = False  #@param {type:"boolean"}
GEE_PROJECT = ""  #@param {type:"string"}
# Tick USE_EARTH_ENGINE and enter your Google Cloud project ID if you have an Earth Engine account.
# You need it for your own area. The three examples work without it.

import os, sys, subprocess, importlib.util, json, re, zipfile, tempfile, shutil, datetime as dt
import xml.etree.ElementTree as ET
from pathlib import Path

REPO_URL = "https://github.com/manudeo/Wetlands_Indaba_2026_workshop.git"
WETLANDMAPPER_VERSION = "1.2.0"
try:
    import google.colab                      # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def find_repo_root():
    # The folder that holds notebooks/ and data/boundaries. Works when the notebook is opened from notebooks/, from the
    # repository root, or from a folder that contains the repository.
    here = Path.cwd().resolve()
    for base in [here, *list(here.parents)[:5]]:
        if (base / "notebooks").is_dir() and (base / "data" / "boundaries").is_dir():
            return base
    for sub in sorted(p for p in here.iterdir() if p.is_dir()):
        if (sub / "notebooks").is_dir() and (sub / "data" / "boundaries").is_dir():
            return sub
    return None

ROOT = find_repo_root()
if ROOT is None:
    if not IN_COLAB:
        raise RuntimeError("Open this notebook from inside the cloned repository (the folder that holds notebooks/ and data/).")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)       # a fresh Colab session
    ROOT = Path.cwd() / "Wetlands_Indaba_2026_workshop"
os.chdir(ROOT)

def have(module):
    try:
        return importlib.util.find_spec(module) is not None
    except (ImportError, ValueError):
        return False

# Packages: pinned and installed in Colab; on your own computer they are checked, never changed.
extra = "[gee]" if USE_EARTH_ENGINE else ""
MODULES = {"wetlandmapper": f"wetlandmapper{extra}=={WETLANDMAPPER_VERSION}", "geopandas": "geopandas", "rioxarray": "rioxarray", "netCDF4": "netCDF4", "scipy": "scipy"}
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *MODULES.values()], check=True)
else:
    missing = [pip for mod, pip in MODULES.items() if not have(mod)]
    if missing:
        raise ImportError("Missing packages: " + ", ".join(missing) + ".\nInstall them in your environment, for example:\n  pip install " + " ".join(f'"{m}"' for m in missing))

import warnings
import numpy as np, pandas as pd, xarray as xr, geopandas as gpd, rioxarray, shapely
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
from rasterio.features import rasterize, shapes
from scipy import ndimage
from scipy.stats import spearmanr
from shapely.geometry import box, shape as shp_shape

import wetlandmapper as wm
from wetlandmapper import (classify_dynamics, classify_wct_ema, compute_wet_frequency, detect_wet_events, build_run_manifest,
                           DYNAMICS_CLASSES, DYNAMICS_COLORS, WCT_CLASSES, WCT_COLORS)
from wetlandmapper.analysis import summarize_dynamics, summarize_wct, class_area_timeseries, summarize_by_polygons, linear_trend
from wetlandmapper.plotting import plot_dynamics, plot_wct

warnings.filterwarnings("ignore", category=FutureWarning); warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11, "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False})
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
CACHE_IN_DRIVE = False  #@param {type:"boolean"}
# On your own computer downloads are kept in data/cache inside the repository and stay there. In Colab that folder is deleted with the session;
# tick CACHE_IN_DRIVE to keep downloads in My Drive/WetlandMapper_cache instead (the repository's sample files are copied there the first time).
CACHE = ROOT / "data" / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
if CACHE_IN_DRIVE:
    if not IN_COLAB:
        print("CACHE_IN_DRIVE is for Colab; on your own computer the cache is", CACHE.resolve())
    else:
        from google.colab import drive
        drive.mount("/content/drive")
        drive_cache = Path("/content/drive/MyDrive/WetlandMapper_cache"); drive_cache.mkdir(parents=True, exist_ok=True)
        for f in CACHE.glob("*"):
            if f.is_file() and not (drive_cache / f.name).exists():
                shutil.copy(f, drive_cache / f.name)
        CACHE = drive_cache
print("Downloads are kept in", CACHE)

GEE_READY = False
if USE_EARTH_ENGINE:
    try:
        from wetlandmapper import gee
        from wetlandmapper.gee import fetch
        try:
            gee.init(project=GEE_PROJECT)            # reuse saved credentials (your own computer, or an earlier sign-in)
        except Exception:
            gee.authenticate(); gee.init(project=GEE_PROJECT)
        GEE_READY = True
        print("Earth Engine ready.")
    except Exception as e:
        print(f"Earth Engine sign-in failed ({type(e).__name__}: {str(e)[:150]}). Check the project ID and that the Earth Engine API is enabled for it.")
else:
    print("Earth Engine is off: the three examples run on cached sample data. Your own area and the live map need Earth Engine.")

def need_gee(what):
    if not GEE_READY:
        raise RuntimeError(f"{what} needs Earth Engine. Tick USE_EARTH_ENGINE in the setup cell, enter your project ID and run it again.")

# ---------------------------------------------------------------- outlines and points
def read_kml_polygons(path):
    # Fallback KML/KMZ reader for when GDAL has no KML driver.
    p = Path(path)
    data = zipfile.ZipFile(p).read(next(n for n in zipfile.ZipFile(p).namelist() if n.lower().endswith(".kml"))) if p.suffix.lower() == ".kmz" else p.read_bytes()
    tag = lambda e: e.tag.split("}")[-1]
    ring = lambda b: [tuple(float(v) for v in t.split(",")[:2]) for t in next(c for c in b.iter() if tag(c) == "coordinates").text.split()]
    polys = []
    for poly in ET.fromstring(data).iter():
        if tag(poly) == "Polygon":
            outer, holes = None, []
            for b in poly:
                if tag(b) == "outerBoundaryIs": outer = ring(b)
                elif tag(b) == "innerBoundaryIs": holes.append(ring(b))
            if outer and len(outer) >= 4: polys.append(shapely.Polygon(outer, holes))
    return gpd.GeoDataFrame({"geometry": polys}, geometry="geometry", crs=4326)

def read_outline(path):
    # Polygons from .kmz, .kml, .gpkg, .geojson, a zipped shapefile or a .shp, as 2-D geometries in EPSG:4326.
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"{p} not found.")
    try:
        g = gpd.read_file(f"zip://{p}") if p.suffix.lower() == ".zip" else gpd.read_file(p)
        ok = g.geom_type.isin(["Polygon", "MultiPolygon"]).any()
    except Exception:
        g, ok = None, False
    if not ok and p.suffix.lower() in (".kml", ".kmz"):
        g = read_kml_polygons(p)
    if g is None or g.crs is None:
        raise ValueError(f"{p.name} has no readable polygons or no coordinate system (a shapefile needs its .prj; or save as GeoPackage).")
    g = g.to_crs(4326)
    g = g.set_geometry(shapely.force_2d(g.geometry.values), crs=4326)
    g = g[g.geom_type.isin(["Polygon", "MultiPolygon"])]
    if g.empty:
        raise ValueError(f"No polygons found in {p.name}: points and lines are not enough. Draw the area as a polygon (Google Earth: Add, Polygon) and save again.")
    return g.set_geometry([shapely.make_valid(x) for x in g.geometry], crs=4326).reset_index(drop=True)

def read_points(path):
    # Field points from a CSV with lat and lon columns, or from a kmz, kml, gpkg, shp or geojson of points.
    p = Path(path)
    if p.suffix.lower() in (".csv", ".txt"):
        df = pd.read_csv(p); low = {c.lower().strip(): c for c in df.columns}
        la = next((low[k] for k in ("lat", "latitude", "y") if k in low), None); lo = next((low[k] for k in ("lon", "lng", "long", "longitude", "x") if k in low), None)
        if la is None or lo is None:
            raise ValueError("The table needs latitude (lat) and longitude (lon) columns in decimal degrees.")
        g = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lo], df[la]), crs=4326)
    else:
        g = gpd.read_file(p).to_crs(4326); g = g.set_geometry(shapely.force_2d(g.geometry.values), crs=4326); g = g[g.geom_type == "Point"]
    g = g.reset_index(drop=True); g.insert(0, "point_id", [f"P{i + 1}" for i in range(len(g))])
    return g

def upload_file():
    # Colab only: open the upload dialog and return the path of the uploaded file (a shapefile also needs its .dbf, .shx and .prj).
    try:
        from google.colab import files
    except ImportError as e:
        raise RuntimeError("Uploading works in Google Colab only. On your own computer choose 'path' and give the full path of the file.") from e
    up = files.upload(); tmp = Path(tempfile.mkdtemp(prefix="wm_upload_"))
    for n, b in up.items(): (tmp / Path(n).name).write_bytes(b)
    order = [".gpkg", ".kmz", ".kml", ".geojson", ".json", ".zip", ".shp", ".csv"]
    found = sorted((q for q in tmp.iterdir() if q.suffix.lower() in order), key=lambda q: order.index(q.suffix.lower()))
    if not found: raise ValueError("No readable file uploaded. Supported: " + ", ".join(order))
    return found[0]

def geojson_of(gdf):
    return shapely.unary_union(gdf.geometry.values).__geo_interface__

def bounds_of(gdf, pad=0.004):
    w, s, e, n = gdf.total_bounds
    return (round(w - pad, 5), round(s - pad, 5), round(e + pad, 5), round(n + pad, 5))

# ---------------------------------------------------------------- the example areas
BOUND = ROOT / "data" / "boundaries"
def example_area(name):
    if name == "agulhas":                                    # the box around the national wetland map polygons of the Agulhas Plain
        g = gpd.read_file(BOUND / "Ramsar_AgulhasPlain_wetlands.shp").to_crs(4326)
        return gpd.GeoDataFrame(geometry=[box(*bounds_of(g, 0.005))], crs=4326)
    if name == "pan":                                        # the study window: the bounding box of pan OBJECTID 3462 with a 1 km buffer
        g = gpd.read_file(BOUND / "Ramsar_AgulhasPlain_wetlands.shp"); g = g[g["OBJECTID"] == 3462]
        return gpd.GeoDataFrame(geometry=[box(*bounds_of(gpd.GeoDataFrame(geometry=g.to_crs(32734).buffer(1000).to_crs(4326).values, crs=4326), 0))], crs=4326)
    if name == "pan_polygon":
        g = gpd.read_file(BOUND / "Ramsar_AgulhasPlain_wetlands.shp"); return g[g["OBJECTID"] == 3462].to_crs(4326).reset_index(drop=True)
    if name == "langebaan":
        return gpd.read_file(BOUND / "Langebaan_Ramsar.shp").to_crs(4326)

def langebaan_box():
    # The request box for Langebaan: the polygon's bounding box plus a little, as in earlier downloads (the data are then masked to the polygon).
    return gpd.GeoDataFrame(geometry=[box(*bounds_of(example_area("langebaan"), 0.003))], crs=4326)

def mask_to(obj, gdf):
    return obj.rio.clip(gdf.geometry, gdf.crs, drop=False, all_touched=True)

# ---------------------------------------------------------------- downloads with WetlandMapper's fetch, and a cache
def _years(obj):
    first = obj if isinstance(obj, xr.DataArray) else obj[list(obj.data_vars)[0]]
    return set(int(y) for y in pd.DatetimeIndex(first["time"].values).year)

def fetch_annual(geom, index, y0, y1, **kw):
    # WetlandMapper's fetch for one composite per year; years skipped by a failed download are fetched again.
    need_gee("Downloading satellite data")
    base = dict(aoi=geom, sensor="LandsatAll", scale=30, temporal_aggregation="annual", use_slc_off=True, project=GEE_PROJECT or None)
    out = fetch(start=f"{y0}-01-01", end=f"{y1}-12-31", index=index, **base, **kw)
    for attempt in range(2):
        miss = sorted(set(range(y0, y1 + 1)) - _years(out))
        if not miss: break
        print(f"  fetching {len(miss)} missing year(s) again: {miss}")
        parts = []
        for y in miss:
            try: parts.append(fetch(start=f"{y}-01-01", end=f"{y}-12-31", index=index, **base, **kw))
            except Exception as e: print(f"    {y}: {type(e).__name__}")
        parts = [q for q in parts if len(_years(q))]
        if parts: out = xr.concat([out] + parts, dim="time").sortby("time")
    return out

def fin(da):
    return da.astype("float32").rio.write_crs("EPSG:4326")

def annual_aweish(geom, y0, y1):
    # Annual AWEIsh composite: the 90th percentile of the cloud-free observations of each year (water is seasonal in dry places).
    da = fetch_annual(geom, "AWEIsh", y0, y1, reduction_method="percentile", percentile=90.0)
    return fin(da["AWEIsh"] if isinstance(da, xr.Dataset) else da)

def wet_season_indices(geom, y0, y1, months):
    # MNDWI, NDVI and NDTI: the median of the rainy-season months of each year.
    ds = fetch_annual(geom, ["MNDWI", "NDVI", "NDTI"], y0, y1, months=list(months), reduction_method="median")
    return xr.Dataset({v: fin(ds[v]) for v in ("MNDWI", "NDVI", "NDTI")})

def save_nc(obj, path):
    # Scaled int16 netCDF, written to a temporary file first so a failed write never leaves a broken file.
    ds = obj.to_dataset() if isinstance(obj, xr.DataArray) else obj
    ds = ds.rio.write_crs("EPSG:4326")
    for v in ds.data_vars:
        for k in ("_FillValue", "missing_value", "scale_factor", "add_offset"): ds[v].attrs.pop(k, None)
        ds[v].encoding.clear()
    enc = {v: {"dtype": "int16", "scale_factor": 0.001, "add_offset": 0.0, "_FillValue": -32768, "zlib": True, "complevel": 5} for v in ds.data_vars}
    tmp = Path(path).with_name(Path(path).stem + ".partial.nc")
    try:
        ds.to_netcdf(tmp, encoding=enc); os.replace(tmp, path)
    finally:
        tmp.unlink(missing_ok=True)

def cached(name, make, force=False):
    # Read data/cache/<name>.nc if it exists; otherwise download with make() (needs Earth Engine) and keep a copy.
    f = CACHE / f"{name}.nc"
    if f.exists() and not force:
        ds = xr.open_dataset(f)
        if len(ds.data_vars) == 0:
            ds.close(); raise FileNotFoundError(f"{f.name} is empty (an interrupted download). Delete it and run the cell again with Earth Engine on.")
        ds = ds.load()
        return fin(ds[list(ds.data_vars)[0]]) if len(ds.data_vars) == 1 else xr.Dataset({v: fin(ds[v]) for v in ds.data_vars})
    if not GEE_READY:
        raise FileNotFoundError(f"The sample data '{name}' are not in data/cache/ yet. Ask the workshop leader, or switch on Earth Engine to download them.")
    print(f"Downloading '{name}' from Earth Engine (kept in data/cache/ for next time)...")
    out = make(); save_nc(out, f); return out

def fill_gaps(name, make_year, polygon=None, y0=1990, y1=2025):
    # Download again only the years that are missing, or hold no valid data, in a cached file (for example after switching on Landsat 7
    # after 2003), and keep every other year as it is.
    f = CACHE / f"{name}.nc"
    if not f.exists():
        return
    old = cached(name, None)
    first = old if isinstance(old, xr.DataArray) else old[list(old.data_vars)[0]]
    inside = inside_mask(polygon, first) if polygon is not None else xr.ones_like(first.isel(time=0, drop=True), dtype=bool)
    yrs = np.asarray(pd.DatetimeIndex(first.time.values).year)
    has = (first.notnull() & inside).any(("y", "x")).values
    bad = sorted(set(range(y0, y1 + 1)) - set(int(y) for y in yrs[has]))
    if not bad:
        print(f"{name}: no gaps"); return
    print(f"{name}: downloading {len(bad)} year(s) again: {bad}")
    fixed = []
    for y in bad:
        try:
            new = make_year(y)
            if polygon is not None: new = mask_to(new, polygon)
            new = new.reindex(y=first.y, x=first.x, method="nearest", tolerance=1e-4)
            ok = new[list(new.data_vars)[0]] if isinstance(new, xr.Dataset) else new
            if bool((ok.notnull() & inside).any()): fixed.append((y, new))
            else: print(f"  {y}: still no usable scenes")
        except Exception as e:
            print(f"  {y}: {type(e).__name__}: {str(e)[:80]}")
    if fixed:
        keep = ~np.isin(yrs, [y for y, _ in fixed])
        merged = xr.concat([old.isel(time=np.where(keep)[0])] + [n for _, n in fixed], dim="time").sortby("time")
        save_nc(merged, f); print(f"  {name}: filled {[y for y, _ in fixed]}")

def rain_monthly(bounds, y0, y1):
    # Monthly rainfall (mm) at the area from ERA5-Land through Earth Engine.
    need_gee("Rainfall")
    import ee
    geom = ee.Geometry.Rectangle(list(bounds))
    col = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR").filterDate(f"{y0}-01-01", f"{y1 + 1}-01-01").select("total_precipitation_sum")
    def f(img):
        d = img.reduceRegion(ee.Reducer.mean(), geom, 11132, maxPixels=int(1e8))
        return ee.Feature(None, {"date": img.date().format("YYYY-MM"), "p": d.get("total_precipitation_sum")})
    feats = ee.FeatureCollection(col.map(f)).getInfo()["features"]
    df = pd.DataFrame([x["properties"] for x in feats]); df["precip_mm"] = df["p"].astype(float) * 1000.0; df["date"] = pd.to_datetime(df["date"] + "-01")
    return df[["date", "precip_mm"]].sort_values("date").reset_index(drop=True)

def cached_rain(name, bounds, y0, y1, force=False):
    f = CACHE / f"{name}.csv"
    if f.exists() and not force: return pd.read_csv(f, parse_dates=["date"])
    if not GEE_READY: raise FileNotFoundError(f"The rainfall table '{name}' is not in data/cache/ yet. Ask the workshop leader, or switch on Earth Engine.")
    df = rain_monthly(bounds, y0, y1); df.to_csv(f, index=False); return df

def suggest_months(rain, width=5):
    # The wettest run of months inside one calendar year (yearly composites are built by calendar year).
    clim = rain.assign(m=rain["date"].dt.month).groupby("m")["precip_mm"].mean().reindex(range(1, 13)).fillna(0)
    best = max(range(1, 13 - width + 1), key=lambda m0: clim.loc[m0:m0 + width - 1].sum())
    wrap = max(range(1, 13), key=lambda m0: sum(clim.loc[((m0 - 1 + k) % 12) + 1] for k in range(width)))
    return list(range(best, best + width)), [((wrap - 1 + k) % 12) + 1 for k in range(width)], clim

# ---------------------------------------------------------------- small numerical helpers
def pixel_area_m2(da):
    dx, dy = (abs(float(r)) for r in da.rio.resolution())
    return dx * 111320 * np.cos(np.radians(float(da.y.mean()))) * dy * 110574

def inside_mask(gdf, like):
    ref = like.isel(time=0, drop=True) if "time" in like.dims else like
    def r(t): return rasterize([(g, 1) for g in gdf.geometry], out_shape=ref.shape, transform=ref.rio.transform(), fill=0, dtype="uint8", all_touched=t).astype(bool)
    m = r(False)
    return xr.DataArray(m if m.any() else r(True), dims=ref.dims, coords=ref.coords)

def utm_epsg(da):
    return (32700 if float(da.y.mean()) < 0 else 32600) + int((float(da.x.mean()) + 180) // 6) + 1

def write_geotiff(da, path, colors):
    # GeoTIFF in the local UTM zone (uint8, 255 = no data) with the class colours embedded, so QGIS shows them without a style file.
    r = max(round(abs(float(da.rio.resolution()[0])) * 111320 * np.cos(np.radians(float(da.y.mean()))) / 5) * 5, 5)
    u = da.rio.write_crs("EPSG:4326").rio.reproject(f"EPSG:{utm_epsg(da)}", resolution=r, resampling=Resampling.nearest)
    arr = u.fillna(255).astype("uint8").values
    with rasterio.open(path, "w", driver="GTiff", height=arr.shape[0], width=arr.shape[1], count=1, dtype="uint8", crs=u.rio.crs, transform=u.rio.transform(), nodata=255, compress="deflate") as dst:
        dst.write(arr, 1)
        dst.write_colormap(1, {**{int(c): tuple(int(h[i:i + 2], 16) for i in (1, 3, 5)) + (255,) for c, h in colors.items()}, 255: (0, 0, 0, 0)})

def wetland_objects(dyn, wf, min_pixels=2):
    # Groups of touching wetland pixels (class above 0) as polygons in UTM with area, dominant class and wet frequency.
    epsg = utm_epsg(dyn); res = max(round(abs(float(dyn.rio.resolution()[0])) * 111320 * np.cos(np.radians(float(dyn.y.mean()))) / 5) * 5, 5)
    utm = xr.Dataset({"cls": dyn.astype("float32"), "wf": wf.astype("float32")}).rio.write_crs("EPSG:4326").rio.reproject(f"EPSG:{epsg}", resolution=res, resampling=Resampling.nearest)
    cls = utm["cls"].values; lab, n = ndimage.label(np.isfinite(cls) & (cls > 0), structure=np.ones((3, 3)))
    if n == 0: return gpd.GeoDataFrame({"obj_id": [], "area_ha": []}, geometry=[], crs=epsg)
    sizes = ndimage.sum(lab > 0, lab, np.arange(1, n + 1)); keep = np.where(sizes >= min_pixels)[0] + 1
    remap = np.zeros(n + 1, dtype=np.int32); remap[keep] = np.arange(1, len(keep) + 1); lab = remap[lab]; ids = np.arange(1, len(keep) + 1)
    if len(ids) == 0: return gpd.GeoDataFrame({"obj_id": [], "area_ha": []}, geometry=[], crs=epsg)
    dom = pd.DataFrame({"o": lab[lab > 0], "c": cls[lab > 0].astype(int)}).groupby("o")["c"].agg(lambda s: s.value_counts().idxmax())
    geoms = [(shp_shape(g), int(v)) for g, v in shapes(lab.astype("int32"), mask=lab > 0, transform=utm.rio.transform(), connectivity=8)]
    gdf = gpd.GeoDataFrame({"obj_id": [v for _, v in geoms]}, geometry=[g for g, _ in geoms], crs=epsg).dissolve(by="obj_id").reset_index()
    gdf["area_ha"] = (ndimage.sum(lab > 0, lab, gdf["obj_id"]) * res * res / 1e4).round(2)
    gdf["dom_class"] = gdf["obj_id"].map(dom).map(DYNAMICS_CLASSES); gdf["wet_freq_pct"] = ndimage.mean(utm["wf"].values, lab, gdf["obj_id"]).round(1)
    return gdf

def write_kmz(gdf, path):
    # Wetland objects for Google Earth or a phone.
    try:
        g = gdf[["obj_id", "area_ha", "dom_class", "geometry"]].to_crs(4326); tmp = Path(tempfile.mkdtemp()) / "doc.kml"; g.to_file(tmp, driver="KML")
        with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z: z.write(tmp, "doc.kml")
    except Exception as e:
        print(f"  (KMZ not written: {type(e).__name__}; the GeoPackage has the same objects)")

def usable_years(ts_ds, mask_px, class_code, frac=0.8):
    # One value per year from class_area_timeseries; years with valid data for under 80% of the area are NaN (no data is not zero).
    d = ts_ds.to_dataframe().reset_index(); d["year"] = pd.to_datetime(d["time"]).dt.year
    tot = d.groupby("year")["total_valid_pixels"].first(); col = [c for c in d.columns if c.startswith("area_")][0]
    s = d[d["class_code"] == class_code].set_index("year")[col]
    return s.where(tot.reindex(s.index) >= frac * mask_px)

def show(df):
    from IPython.display import display
    display(df)

print("WetlandMapper", wm.__version__, "|", "Colab" if IN_COLAB else "local", "| repository:", ROOT)

## Your own outline (optional, do this when you are ready)

An outline is a **polygon** of your wetland or study area: KMZ or KML (drawn in Google Earth: Add, Polygon), GeoPackage, GeoJSON, a zipped shapefile, or a shapefile with its `.dbf`, `.shx` and `.prj`. Points and lines are not enough.

Run this cell once. Then, in Part A, B or C, set *AREA* to **my own outline** and run that part again. If you have no outline of your own, choose the example KMZ (the pan from Part B) to test the route.

In [ ]:
#@title Your own outline { display-mode: "form" }
AREA_NAME = "my_wetland"  #@param {type:"string"}
OUTLINE = "upload"  #@param ["upload", "path in Drive or Colab", "example: pan 3462 (KMZ)", "none"]
OUTLINE_PATH = ""  #@param {type:"string"}
if OUTLINE == "none":
    own_aoi = None
else:
    own_path = upload_file() if OUTLINE == "upload" else (OUTLINE_PATH if OUTLINE == "path in Drive or Colab" else ROOT / "data" / "examples" / "pan3462_outline.kmz")
    own_aoi = read_outline(own_path)
    c = shapely.unary_union(own_aoi.geometry.values).centroid
    ha = own_aoi.to_crs((32700 if c.y < 0 else 32600) + int((c.x + 180) // 6) + 1).area.sum() / 1e4
    nx = int((own_aoi.total_bounds[2] - own_aoi.total_bounds[0]) / (30 / 111319.49)); ny = int((own_aoi.total_bounds[3] - own_aoi.total_bounds[1]) / (30 / 111319.49))
    print(f"Outline '{AREA_NAME}': {len(own_aoi)} polygon(s), {ha:,.1f} ha, box {nx} x {ny} pixels at 30 m")
    if nx * ny > 2.5e6: raise ValueError("The outline's box is too large for this notebook (over about 2.5 million pixels per year). Use a smaller outline.")
    fig, ax = plt.subplots(figsize=(4.5, 4)); own_aoi.plot(ax=ax, color="#76d7c4", edgecolor="k"); ax.set_title(AREA_NAME); ax.set_aspect(1 / np.cos(np.radians(float(own_aoi.total_bounds[[1, 3]].mean())))); plt.show()

## Part A. Find the wetlands in an area: the Agulhas Plain

**The question.** Which parts of this landscape behave like wetland, and how does that compare with the national wetland map?

WetlandMapper reads 36 yearly water-index images (1990 to 2025). A pixel that is wet often enough counts as wetland. Groups of touching wetland pixels become **wetland objects**: polygons with an area, a dominant class and a wet frequency. For the example, the result is compared with the mapped wetland polygons of the Agulhas Plain (National Wetland Map 5, as supplied by SANParks).

Two sliders: **minimum wet years** is how often a pixel must be wet to count (25% by default); **minimum object size** drops specks.

In [ ]:
#@title Part A. Find the wetlands { display-mode: "form" }
AREA = "example: Agulhas Plain"  #@param ["example: Agulhas Plain", "my own outline"]
MINIMUM_WET_YEARS_PERCENT = 25  #@param {type:"slider", min:5, max:60, step:5}
MINIMUM_OBJECT_SIZE_PIXELS = 2  #@param {type:"slider", min:1, max:10, step:1}
FIRST_YEAR_A = 1990  #@param {type:"integer"}
LAST_YEAR_A = 2025  #@param {type:"integer"}

example_a = AREA.startswith("example")
if example_a:
    aoi_a, name_a, ref = example_area("agulhas"), "agulhas_plain", gpd.read_file(BOUND / "Ramsar_AgulhasPlain_wetlands.shp").to_crs(4326)
    aw_a = cached("agulhas_aweish_annual", lambda: annual_aweish(geojson_of(aoi_a), 1990, 2025))
else:
    assert own_aoi is not None, "Run the 'Your own outline' cell first."
    aoi_a, name_a, ref = own_aoi, AREA_NAME + "_find", None
    aw_a = annual_aweish(geojson_of(aoi_a), FIRST_YEAR_A, LAST_YEAR_A)
out_a = OUT / name_a; out_a.mkdir(exist_ok=True)

mask_a = inside_mask(aoi_a, aw_a); ok_a = aw_a.notnull().any("time") & mask_a
dyn_a = classify_dynamics(aw_a, nYear=3, thresholdWet=MINIMUM_WET_YEARS_PERCENT, thresholdPersis=75, nan_policy="valid").where(ok_a)
wf_a = compute_wet_frequency(aw_a, nan_policy="valid").where(ok_a)
px_a = pixel_area_m2(aw_a)
classes_a = summarize_dynamics(dyn_a, pixel_area=px_a, area_unit="ha").to_dataframe().drop(columns=["class_name"]).join(pd.Series(DYNAMICS_CLASSES, name="class"))
objs_a = wetland_objects(dyn_a, wf_a, MINIMUM_OBJECT_SIZE_PIXELS)
wet_ha = float(classes_a.loc[classes_a.index != 0, "area_ha"].sum()); tot_ha = float(ok_a.sum()) * px_a / 1e4
print(f"Wetland found: {wet_ha:,.0f} ha of {tot_ha:,.0f} ha analysed ({100 * wet_ha / tot_ha:.1f}%), in {len(objs_a):,} objects of at least {MINIMUM_OBJECT_SIZE_PIXELS} pixels.")
if len(objs_a): print(f"Median object {objs_a['area_ha'].median():.2f} ha; largest {objs_a['area_ha'].max():,.1f} ha.")
yrs_a = sorted(set(pd.DatetimeIndex(aw_a.time.values).year)); gaps_a = [y for y in range(yrs_a[0], yrs_a[-1] + 1) if y not in yrs_a]
if gaps_a: print(f"No composite for {gaps_a}: those years are left out.")
show(classes_a[["class", "pixel_count", "area_ha", "percent_of_valid"]])

fig, ax = plot_dynamics(dyn_a, title=f"{name_a}: wetland dynamics {yrs_a[0]} to {yrs_a[-1]}", figsize=(11, 5.4))
if ref is not None: ref.boundary.plot(ax=ax, color="k", linewidth=0.4)
ax.set_aspect(1 / np.cos(np.radians(float(dyn_a.y.mean())))); fig.savefig(out_a / f"{name_a}_dynamics_map.png", dpi=150, bbox_inches="tight"); plt.show()

if ref is not None:                                           # compare with the mapped layer, by wetland type
    ref = ref.reset_index(drop=True).assign(poly=lambda d: d.index)
    flagged = (dyn_a > 0).where(dyn_a.notnull()).astype("float32")
    z = summarize_by_polygons(flagged, ref[["poly", "geometry"]], polygon_id_col="poly", stats=("mean", "count")).merge(ref[["poly", "Ramsar"]], left_on="polygon_id", right_on="poly")
    z["flagged_px"] = z["mean"] * z["count"]
    by_type = z.groupby("Ramsar").agg(mapped_ha=("count", lambda c: c.sum() * px_a / 1e4), flagged_ha=("flagged_px", lambda c: c.sum() * px_a / 1e4))
    by_type.insert(0, "polygons", ref.groupby("Ramsar").size())          # all polygons; those smaller than a pixel add no area
    by_type["flagged_pct"] = (100 * by_type["flagged_ha"] / by_type["mapped_ha"]).round(0); by_type = by_type.round(1)
    cand = objs_a[~objs_a.intersects(shapely.unary_union(ref.to_crs(objs_a.crs).geometry.values))].sort_values("area_ha", ascending=False)
    print(f"\nShare of the mapped layer flagged as wetland: {100 * by_type['flagged_ha'].sum() / by_type['mapped_ha'].sum():.0f}% overall. "
          f"{len(cand)} wetland objects ({cand['area_ha'].sum():,.0f} ha) do not touch the mapped layer.")
    show(by_type)
    fig, ax = plt.subplots(figsize=(7.5, 3.2)); t = by_type.sort_values("mapped_ha")
    ax.barh(t.index, t["flagged_pct"], color="#1a5276"); ax.set_xlim(0, 115); ax.set_xlabel("% of mapped area flagged as wetland"); ax.set_title("By wetland type (National Wetland Map 5)", fontsize=10)
    for i, (v, n) in enumerate(zip(t["flagged_pct"], t["polygons"])): ax.text(v + 1, i, f"{v:.0f}%  (n = {n})", va="center", fontsize=8)
    fig.tight_layout(); fig.savefig(out_a / f"{name_a}_vs_mapped_layer.png", dpi=150, bbox_inches="tight"); plt.show()
    by_type.to_csv(out_a / f"{name_a}_vs_mapped_layer.csv"); cand.drop(columns="geometry").head(25).to_csv(out_a / f"{name_a}_candidates_not_in_mapped_layer.csv", index=False)
    print("Largest objects not touching the mapped layer (places to look):"); show(cand.drop(columns="geometry").head(8).set_index("obj_id"))

write_geotiff(dyn_a, out_a / f"{name_a}_dynamics.tif", DYNAMICS_COLORS)
classes_a.to_csv(out_a / f"{name_a}_dynamics_classes.csv")
if len(objs_a):
    objs_a.to_file(out_a / f"{name_a}_wetland_objects.gpkg", layer="wetland_objects", driver="GPKG"); write_kmz(objs_a, out_a / f"{name_a}_wetland_objects.kmz")
build_run_manifest(parameters={"area": name_a, "nYear": 3, "thresholdWet": MINIMUM_WET_YEARS_PERCENT, "thresholdPersis": 75, "min_object_pixels": MINIMUM_OBJECT_SIZE_PIXELS,
                               "composite": "AWEIsh annual 90th percentile", "sensor": "LandsatAll incl. Landsat 7 after 2003", "years": [yrs_a[0], yrs_a[-1]]},
                   output_path=out_a / f"{name_a}_run_manifest.json")
print("files:", out_a)

**How to read it**

- **The sentences at the top** are the headline: how much wetland, in how many objects, and how much of the mapped layer it covers.
- **The map** colours every wetland pixel by how it behaved. *Persistent* and *Intermittent* are the stable part; *New* and *Lost* changed between the first and last years.
- **The table by wetland type** shows where the method and the map agree. A type with a low percentage is not necessarily wrong: seeps are wet at the surface only briefly, and a 30 m, yearly, open-water record sees them poorly.
- **Objects that do not touch the mapped layer** are places to look, not new wetlands: check them against imagery or the field.
- Change the sliders and run again to see how much the answer depends on the thresholds. Report them with any map.

Files are in `outputs/agulhas_plain/`: a GeoTIFF with the class colours embedded, the wetland objects as GeoPackage, KMZ (Google Earth) and CSV, the tables, the figures, and a run manifest.

## Part B. One wetland through 36 years

**The question.** For a wetland we already know (a saline pan, OBJECTID 3462), how have its wetness and cover changed, and does rainfall explain it?

This is the report you can run for any outline: wet area per year, a trend, the link to rainfall, dynamics classes, and cover types through time. The cover types come from composites of the rainy-season months (June to October for the example). For your own area the notebook suggests the months from the rainfall at your site: the wet season of the Western Cape (about June to October) is not that of Gauteng, KwaZulu-Natal or Lesotho (about October to March). A season that spans New Year cannot be one yearly composite, so the notebook picks the best five months inside one calendar year.

In [ ]:
#@title Part B. One wetland through time { display-mode: "form" }
AREA = "example: pan 3462"  #@param ["example: pan 3462", "my own outline"]
MINIMUM_WET_YEARS_PERCENT = 25  #@param {type:"slider", min:5, max:60, step:5}
RAINY_SEASON_MONTHS = "auto"  #@param {type:"string"}
FIRST_YEAR_B = 1990  #@param {type:"integer"}
LAST_YEAR_B = 2025  #@param {type:"integer"}

example_b = AREA.startswith("example")
if example_b:
    aoi_b, window_b, name_b = example_area("pan_polygon"), example_area("pan"), "pan3462"
    y0b, y1b, months_b = 1990, 2025, [6, 7, 8, 9, 10]
    rain_b = cached_rain("pan3462_era5_monthly", bounds_of(window_b, 0), 1990, 2025)
    aw_b = cached("pan3462_aweish_annual", lambda: annual_aweish(geojson_of(window_b), 1990, 2025))
    wct_b = cached("pan3462_wct_wetseason", lambda: wet_season_indices(geojson_of(window_b), 1990, 2025, months_b))
else:
    assert own_aoi is not None, "Run the 'Your own outline' cell first."
    aoi_b, name_b, y0b, y1b = own_aoi, AREA_NAME, FIRST_YEAR_B, LAST_YEAR_B
    rain_b = rain_monthly(bounds_of(aoi_b), y0b, y1b)
    best, wrap, clim = suggest_months(rain_b)
    months_b = best if RAINY_SEASON_MONTHS.strip().lower() == "auto" else [int(m) for m in RAINY_SEASON_MONTHS.replace(" ", "").split(",") if m]
    print("Rainy-season months:", months_b, "(from rainfall at your site)" if RAINY_SEASON_MONTHS.strip().lower() == "auto" else "(yours)")
    if RAINY_SEASON_MONTHS.strip().lower() == "auto" and wrap[0] > wrap[-1]:
        print(f"Note: the five wettest months run across New Year ({wrap}); a yearly composite cannot use them, so the window above is the best inside one calendar year.")
    fig, ax = plt.subplots(figsize=(7, 3)); ax.bar(clim.index, clim.values, color=["#e67e22" if m in months_b else "#1a5276" for m in clim.index])
    ax.set_xticks(range(1, 13)); ax.set_xticklabels(list("JFMAMJJASOND")); ax.set_ylabel("mm per month"); ax.set_title("Typical monthly rainfall (orange = chosen season)", fontsize=10); plt.show()
    aw_b = annual_aweish(geojson_of(aoi_b), y0b, y1b); wct_b = wet_season_indices(geojson_of(aoi_b), y0b, y1b, months_b)
out_b = OUT / name_b; out_b.mkdir(exist_ok=True)

w, s, e, n = aoi_b.total_bounds; pad = 0.0045
aw_b = aw_b.rio.clip_box(w - pad, s - pad, e + pad, n + pad); wct_b = wct_b.rio.clip_box(w - pad, s - pad, e + pad, n + pad)
mask_b = inside_mask(aoi_b, aw_b); px_b = pixel_area_m2(aw_b); mpx = int(mask_b.sum()); area_ha_b = mpx * px_b / 1e4
ok_b = aw_b.notnull().any("time")
dyn_b = classify_dynamics(aw_b, nYear=3, thresholdWet=MINIMUM_WET_YEARS_PERCENT, thresholdPersis=75, nan_policy="valid").where(ok_b)
wf_b = compute_wet_frequency(aw_b, nan_policy="valid").where(ok_b)

# wet area by year: class_area_timeseries on a wet/dry raster; years with little valid data are NaN, not zero
wetdry = (aw_b > 0).where(aw_b.notnull()).where(mask_b).astype("float32")
wet_ts = usable_years(class_area_timeseries(wetdry, class_labels={0: "dry", 1: "wet"}, pixel_area=px_b, area_unit="ha"), mpx, 1)
gaps_b = [y for y in range(int(wet_ts.index.min()), int(wet_ts.index.max()) + 1) if y not in set(wet_ts.dropna().index)]
ws = wet_ts.dropna()
sl = linear_trend(xr.DataArray(ws.values, dims="time", coords={"time": pd.to_datetime([f"{y}-07-01" for y in ws.index])}, name="wet_ha"))
k = min(5, max(len(ws) // 3, 1))
rain_y = None
if rain_b is not None and len(ws) >= 8:
    ep = rain_b.set_index("date")["precip_mm"]
    rain_y = pd.Series({y: ep[pd.Timestamp(y - 1, 11, 1): pd.Timestamp(y, 10, 1)].sum() for y in ws.index})
    rho, pval = spearmanr(rain_y.values, ws.values)
lines = [f"{name_b}: mapped area {area_ha_b:,.1f} ha.",
         f"Wet area (water index above 0) averaged {ws.mean():,.1f} ha a year, from {ws.min():,.1f} ha in {ws.idxmin()} to {ws.max():,.1f} ha in {ws.idxmax()}.",
         f"The mean of the last {k} years ({ws.iloc[-k:].mean():,.1f} ha) is {abs(100 * (ws.iloc[-k:].mean() / ws.iloc[:k].mean() - 1)):.0f}% {'higher' if ws.iloc[-k:].mean() >= ws.iloc[:k].mean() else 'lower'} than the first {k} ({ws.iloc[:k].mean():,.1f} ha).",
         f"Linear trend {float(sl['slope']) * 10:+.1f} ha per decade (p = {float(sl['p_value']):.2f}): " + ("a consistent direction." if float(sl['p_value']) < 0.05 else "no clear trend. Read it as a direction over noisy yearly values, not a precise rate.")]
if rain_y is not None: lines.append(f"Wet area against the previous 12 months of rainfall: Spearman rho = {rho:+.2f} (p = {pval:.3f}, {len(ws)} years).")
if gaps_b: lines.append(f"Years with no usable data inside the outline, left out of the statistics: {', '.join(map(str, gaps_b))}.")
lines.append(f"Inside the outline, {float(summarize_dynamics(dyn_b.where(mask_b), pixel_area=px_b, area_unit='ha')['area_ha'].sel(class_code=slice(2, 10)).sum()):,.1f} ha is classed as wetland.")
print("\n".join("- " + l for l in lines))

classes_b = summarize_dynamics(dyn_b.where(mask_b), pixel_area=px_b, area_unit="ha").to_dataframe().drop(columns=["class_name"]).join(pd.Series(DYNAMICS_CLASSES, name="class"))
show(classes_b[["class", "pixel_count", "area_ha"]])

fig, axs = plt.subplots(1, 2, figsize=(13, 5.2))
plot_dynamics(dyn_b, ax=axs[0], title="Dynamics", legend_loc="lower right"); wf_b.plot.imshow(ax=axs[1], vmin=0, vmax=100, cmap="Blues", cbar_kwargs={"label": "% of years wet"}); axs[1].set_title("Wet frequency")
for ax in axs: aoi_b.boundary.plot(ax=ax, color="k", linewidth=1.2); ax.set_aspect(1 / np.cos(np.radians(float(aw_b.y.mean())))); ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
fig.savefig(out_b / f"{name_b}_dynamics_maps.png", dpi=150, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(11, 3.8)); ax.bar(wet_ts.index, wet_ts.values, color="#1a5276")
for y in gaps_b: ax.axvspan(y - 0.5, y + 0.5, color="#e5e8ea", zorder=0)
ax.axhline(area_ha_b, color="#c0392b", ls="--", lw=1, label="mapped area"); ax.legend(loc="upper left", fontsize=8); ax.set_ylabel("wet area (ha)"); ax.set_title(f"{name_b}: wet area by year" + (" (grey = no usable data)" if gaps_b else ""))
if rain_y is not None:
    ax2 = ax.twinx(); ax2.spines["right"].set_visible(True); ax2.plot(rain_y.index, rain_y.values, color="#e67e22", marker="o", ms=3); ax2.set_ylabel("rain, previous 12 months (mm)", color="#e67e22")
fig.tight_layout(); fig.savefig(out_b / f"{name_b}_wet_area_by_year.png", dpi=150, bbox_inches="tight"); plt.show()
wet_ts.rename("wet_area_ha").to_csv(out_b / f"{name_b}_wet_area_by_year.csv")

ev = detect_wet_events(aw_b, threshold=0.0)
fig, axs = plt.subplots(1, 3, figsize=(15, 4.3))
for ax, v, lab in ((axs[0], "first_on_year", "first wet year"), (axs[1], "last_on_year", "last wet year"), (axs[2], "longest_on_streak", "longest run of wet years")):
    ev[v].where(ok_b).plot.imshow(ax=ax, cmap="viridis" if v != "longest_on_streak" else "magma_r", cbar_kwargs={"label": lab}); aoi_b.boundary.plot(ax=ax, color="k", linewidth=1); ax.set_title(lab)
    ax.set_aspect(1 / np.cos(np.radians(float(aw_b.y.mean())))); ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude" if v == "first_on_year" else "")
fig.savefig(out_b / f"{name_b}_wet_events.png", dpi=150, bbox_inches="tight"); plt.show()

# cover types through time. Missing pixels are put back to NaN after classifying (older versions return class 0 for them).
valid_b = wct_b[["MNDWI", "NDVI", "NDTI"]].to_array().notnull().all("variable")
wct_cls = classify_wct_ema(wct_b[["MNDWI", "NDVI", "NDTI"]])["wetland_cover_type"].where(valid_b)
pxw = pixel_area_m2(wct_b["MNDWI"]); maskw = inside_mask(aoi_b, wct_b["MNDWI"]); mpxw = int(maskw.sum())
wts = class_area_timeseries(wct_cls.where(maskw), class_labels=WCT_CLASSES, pixel_area=pxw, area_unit="ha")
area_w = pd.DataFrame({WCT_CLASSES[c]: usable_years(wts, mpxw, c) for c in WCT_CLASSES}); area_w.index.name = "year"
cols = [WCT_CLASSES[c] for c in range(1, 6)]
fig, ax = plt.subplots(figsize=(11, 3.8)); ax.stackplot(area_w.index, [area_w[c].fillna(0) for c in cols], labels=cols, colors=[WCT_COLORS[c] for c in range(1, 6)])
for y in area_w.index[area_w.isna().all(axis=1)]: ax.axvspan(y - 0.5, y + 0.5, color="#e5e8ea", zorder=0)
ax.legend(loc="upper left", fontsize=7, ncol=3); ax.set_ylabel("area inside the outline (ha)"); ax.set_title(f"{name_b}: cover types by year (rainy-season composite, months {months_b}; grey = no data)")
fig.tight_layout(); fig.savefig(out_b / f"{name_b}_cover_types_by_year.png", dpi=150, bbox_inches="tight"); plt.show()
area_w.round(2).to_csv(out_b / f"{name_b}_cover_types_by_year.csv")
av = area_w.dropna(how="all"); kk = min(5, max(len(av) // 3, 1))
chg = pd.DataFrame({f"first {kk} yrs (ha)": av.iloc[:kk].mean().round(1), f"last {kk} yrs (ha)": av.iloc[-kk:].mean().round(1)}); chg["change (ha)"] = chg.iloc[:, 1] - chg.iloc[:, 0]
print("Cover-type area, first versus last years:"); show(chg); chg.to_csv(out_b / f"{name_b}_cover_types_change.csv")
sel = sorted(set([int(av.index.min()), int(av.index.max()), int(ws.idxmin()), int(ws.idxmax())]))
fig, axs = plt.subplots(1, len(sel), figsize=(4.3 * len(sel), 4.4), squeeze=False)
for ax, y in zip(axs[0], sel):
    plot_wct(wct_cls.isel(time=int(np.argmax(pd.DatetimeIndex(wct_cls.time.values).year == y))), ax=ax, title=str(y), add_colorbar=(y == sel[-1])); aoi_b.boundary.plot(ax=ax, color="k", linewidth=1); ax.set_aspect(1 / np.cos(np.radians(float(wct_b.y.mean()))))
fig.savefig(out_b / f"{name_b}_cover_type_maps.png", dpi=150, bbox_inches="tight"); plt.show()

# per polygon, and files
aoi_b = aoi_b.reset_index(drop=True).assign(poly=lambda d: d.index)
per = summarize_by_polygons(wf_b, aoi_b[["poly", "geometry"]], polygon_id_col="poly", stats=("mean", "count")).rename(columns={"mean": "mean_wet_frequency_pct", "count": "pixels"})
per["area_ha"] = (per["pixels"] * px_b / 1e4).round(2); per.to_csv(out_b / f"{name_b}_per_polygon.csv", index=False)
if len(per) > 1: print("Statistics per polygon:"); show(per)
write_geotiff(dyn_b, out_b / f"{name_b}_dynamics.tif", DYNAMICS_COLORS)
write_geotiff(wct_cls.isel(time=-1), out_b / f"{name_b}_cover_types_{int(pd.DatetimeIndex(wct_cls.time.values).year[-1])}.tif", WCT_COLORS)
classes_b.to_csv(out_b / f"{name_b}_dynamics_classes.csv"); (out_b / f"{name_b}_report.md").write_text(f"# {name_b}\n\n" + "\n".join("- " + l for l in lines) + "\n")
build_run_manifest(parameters={"area": name_b, "nYear": 3, "thresholdWet": MINIMUM_WET_YEARS_PERCENT, "thresholdPersis": 75, "years": [y0b, y1b], "wet_season_months": months_b,
                               "dynamics_composite": "AWEIsh annual 90th percentile", "cover_type_composite": "MNDWI, NDVI, NDTI median of the rainy-season months", "rainfall": "ERA5-Land monthly"},
                   output_path=out_b / f"{name_b}_run_manifest.json")
print("files:", out_b)

**How to read it**

- **Wet area by year**, with rainfall on the same chart. If the bars follow the line, rainfall is the main driver; the correlation in the sentences says how closely.
- **The trend sentence is a direction, not a precise rate.** A wetland driven by rainfall can show a trend only because the record starts in a wet or dry run.
- **Grey years had no usable data.** Landsat coverage can be thin in some years (the example has a gap around 2009 to 2012 because few clear scenes are available there). They are left out of the statistics, not counted as dry.
- **Dynamics and wet-frequency maps** show which parts are always wet and which fill only in wet years. **Wet-event maps** show when each pixel was first and last wet.
- **Cover types by year** show whether open water, vegetation or moist soil changes with wetness, and the maps pick the first and last years and the driest and wettest.

The mapped polygon is the best map available, not ground truth: its validation fields are empty.

## Part C. Cover types this week: the Langebaan Ramsar site

**The question.** What cover types does the satellite see at Langebaan right now, so you can compare with your own field visit?

Sentinel-2 passes about every five days. The notebook takes the scenes of the last days, drops the cloudy ones, builds a median composite, and classifies it into cover types. It lists the scenes with their **date and time in South African time**, so you can match them to your visit.

**Two cautions.** Langebaan is **tidal**: the image shows the tide at the moment of the overpass, which may differ from the tide at your visit. And the classes are spectral: *Submerged* and *Emergent vegetation* describe what the indices look like, not a verified habitat. Your field notes are the check.

- With Earth Engine on, this is **live**. Without it you see a sample snapshot downloaded shortly before the workshop; the cell says which.
- Add your **field points** (a CSV with `lat` and `lon` columns, or a KMZ, KML or GeoPackage of points) to get the cover type and the index values at each point.

In [ ]:
#@title Part C. Cover types this week { display-mode: "form" }
AREA = "example: Langebaan Ramsar site"  #@param ["example: Langebaan Ramsar site", "my own outline"]
DAYS_BACK = 14  #@param {type:"slider", min:5, max:30, step:1}
MINIMUM_VALID_PERCENT = 40  #@param {type:"slider", min:10, max:90, step:5}
FIELD_POINTS = "none"  #@param ["none", "upload now", "path in Drive or Colab"]
FIELD_POINTS_PATH = ""  #@param {type:"string"}

if AREA.startswith("example"):
    aoi_c, name_c = example_area("langebaan"), "langebaan"
else:
    assert own_aoi is not None, "Run the 'Your own outline' cell first."
    aoi_c, name_c = own_aoi, AREA_NAME
out_c = OUT / (name_c + "_now"); out_c.mkdir(exist_ok=True)
end = dt.date.today() + dt.timedelta(days=1); start = end - dt.timedelta(days=DAYS_BACK + 1)
if GEE_READY:
    ds_c = fetch(aoi=geojson_of(aoi_c), start=start.isoformat(), end=end.isoformat(), sensor="Sentinel2", index=["MNDWI", "NDVI", "NDTI"], scale=20, temporal_aggregation="all", max_cloud_cover=60.0, project=GEE_PROJECT or None)
    ds_c = xr.Dataset({v: fin(ds_c[v]) for v in ("MNDWI", "NDVI", "NDTI")}); source_c = f"live from Earth Engine, {dt.date.today()}"
elif AREA.startswith("example") and (CACHE / "langebaan_s2_recent.nc").exists():
    ds_c = cached("langebaan_s2_recent", None); meta = json.loads((CACHE / "langebaan_recent_meta.json").read_text()) if (CACHE / "langebaan_recent_meta.json").exists() else {}
    source_c = f"sample snapshot downloaded {meta.get('downloaded') or meta.get('exported', 'earlier')}; switch on Earth Engine for a live map"
else:
    need_gee("A recent map of your own area")
print("Data:", source_c)

mask_c = inside_mask(aoi_c, ds_c["MNDWI"]); t = pd.DatetimeIndex(ds_c.time.values)
valid_all = ds_c[["MNDWI", "NDVI", "NDTI"]].to_array().notnull().all("variable") & mask_c
scenes = pd.DataFrame({"time_utc": t, "time_local": t.tz_localize("UTC").tz_convert("Africa/Johannesburg").tz_localize(None), "valid_pct": (100 * valid_all.sum(("x", "y")) / int(mask_c.sum())).values.round(1)})
scenes["used"] = scenes["valid_pct"] >= MINIMUM_VALID_PERCENT
if not scenes["used"].any(): raise ValueError(f"No scene has at least {MINIMUM_VALID_PERCENT}% valid pixels inside the outline. Increase DAYS_BACK, lower the minimum, or try after the next clear satellite pass.")
sel_c = ds_c.isel(time=np.where(scenes["used"].values)[0])
comp = xr.Dataset({v: sel_c[v].median("time", skipna=True) for v in ("MNDWI", "NDVI", "NDTI")}).rio.write_crs("EPSG:4326")
vc = comp[["MNDWI", "NDVI", "NDTI"]].to_array().notnull().all("variable") & mask_c
wct_now = classify_wct_ema(comp)["wetland_cover_type"].where(vc)
px_c = pixel_area_m2(comp["MNDWI"])
now_tab = summarize_wct(wct_now.where(mask_c), pixel_area=px_c, area_unit="ha").to_dataframe()[["class_name", "pixel_count", "area_ha"]]
f0, f1 = scenes.loc[scenes["used"], "time_local"].min(), scenes.loc[scenes["used"], "time_local"].max()
print(f"{int(scenes['used'].sum())} of {len(scenes)} scenes used (at least {MINIMUM_VALID_PERCENT}% valid), from {f0:%d %b %Y %H:%M} to {f1:%d %b %Y %H:%M} South African time. {100 * float(vc.sum() / mask_c.sum()):.0f}% of the outline has a valid composite.")
show(scenes); show(now_tab)

# context: the usual rainy-season cover types, 2016 to 2025 (Landsat, 30 m, a different sensor and season: context only)
if AREA.startswith("example"):
    usual_ds = cached("langebaan_wct_wetseason", lambda: mask_to(wet_season_indices(geojson_of(langebaan_box()), 1990, 2025, [6, 7, 8, 9, 10]), aoi_c))
    uv = usual_ds[["MNDWI", "NDVI", "NDTI"]].to_array().notnull().all("variable")
    ucls = classify_wct_ema(usual_ds[["MNDWI", "NDVI", "NDTI"]])["wetland_cover_type"].where(uv).isel(time=slice(-10, None))
    um = inside_mask(aoi_c, usual_ds["MNDWI"]); uts = class_area_timeseries(ucls.where(um), class_labels=WCT_CLASSES, pixel_area=pixel_area_m2(usual_ds["MNDWI"]), area_unit="ha")
    usual = pd.Series({WCT_CLASSES[c]: usable_years(uts, int(um.sum()), c).mean() for c in WCT_CLASSES})
    cmp_ = pd.DataFrame({"now (ha)": now_tab["area_ha"].values, "usual rainy season, 2016 to 2025 (ha)": usual.values.round(1)}, index=now_tab["class_name"].values); cmp_["difference (ha)"] = (cmp_.iloc[:, 0] - cmp_.iloc[:, 1]).round(1)
    print("\nCompared with the usual rainy season. Different sensor, season and resolution: use as context only."); show(cmp_); cmp_.to_csv(out_c / f"{name_c}_now_vs_usual.csv")

fig, ax = plot_wct(wct_now, title=f"{name_c}: cover types, {f0:%d %b} to {f1:%d %b %Y}", figsize=(8, 7)); aoi_c.boundary.plot(ax=ax, color="k", linewidth=0.5); ax.set_aspect(1 / np.cos(np.radians(float(wct_now.y.mean()))))
pts_path = upload_file() if FIELD_POINTS == "upload now" else (FIELD_POINTS_PATH if FIELD_POINTS == "path in Drive or Colab" else None)
if pts_path:
    pts = read_points(pts_path); rows = []
    for r in pts.itertuples():
        iy, ix = int(np.abs(wct_now.y.values - r.geometry.y).argmin()), int(np.abs(wct_now.x.values - r.geometry.x).argmin()); c = wct_now.values[iy, ix]
        rows.append({"point_id": r.point_id, "lon": round(r.geometry.x, 5), "lat": round(r.geometry.y, 5), "cover type now": WCT_CLASSES.get(int(c), "") if np.isfinite(c) else "no valid data",
                     **{v: round(float(comp[v].values[iy, ix]), 2) for v in ("MNDWI", "NDVI", "NDTI")}})
        ax.plot(r.geometry.x, r.geometry.y, "k^", ms=7, mfc="none")
    ptab = pd.DataFrame(rows); print("\nValues at your field points (write down the tide at your visit):"); show(ptab); ptab.to_csv(out_c / f"{name_c}_field_points_now.csv", index=False)
fig.savefig(out_c / f"{name_c}_cover_types_now.png", dpi=150, bbox_inches="tight"); plt.show()
scenes.to_csv(out_c / f"{name_c}_scenes.csv", index=False); now_tab.to_csv(out_c / f"{name_c}_cover_types_now.csv")
write_geotiff(wct_now, out_c / f"{name_c}_cover_types_now.tif", WCT_COLORS)
build_run_manifest(parameters={"area": name_c, "sensor": "Sentinel-2 SR harmonized", "scale_m": 20, "window": f"{start} to {end}", "min_valid_percent": MINIMUM_VALID_PERCENT, "composite": "median of used scenes", "data": source_c},
                   output_path=out_c / f"{name_c}_now_run_manifest.json")
print("files:", out_c)

**How to read it**

- **The scene list** shows which images went into the map and when they were taken (South African time). A scene with few valid pixels is mostly cloud and is left out.
- **The map and class table** give the cover types now. **The comparison with the usual rainy season** is context only: Landsat composites from June to October, 2016 to 2025, a different sensor, season and resolution.
- **Field points:** the table gives the class and the three index values (MNDWI for open water, NDVI for green vegetation, NDTI for turbidity). If your notes and the class disagree, the index values tell you which way the pixel leans.
- If the map looks wrong, check the scene dates first: haze, cloud or a different tide is the usual reason.

## Keep your results

Everything is written to the `outputs` folder, which Colab deletes when the session ends. Download a zip, or copy it to a Google Drive folder.

In [ ]:
#@title Download or save your results { display-mode: "form" }
SAVE_TO = "zip in the outputs folder"  #@param ["zip in the outputs folder", "download zip (Colab)", "copy to Google Drive (Colab)", "copy to a folder"]
FOLDER = ""  #@param {type:"string"}
# Google Drive: a folder name inside My Drive (default WetlandMapper_results). Copy to a folder: any folder path on your computer.

DRIVE_ROOT = Path("/content/drive/MyDrive")
zip_path = Path(shutil.make_archive(str(ROOT / "wetlandmapper_results"), "zip", OUT))     # the zip sits next to outputs/, not inside it
print("Zipped results:", zip_path)

def copy_results(dest):
    dest = Path(dest).expanduser(); dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree(OUT, dest / "outputs", dirs_exist_ok=True); shutil.copy(zip_path, dest / zip_path.name)
    print("Copied the outputs folder and the zip to", dest.resolve())

if SAVE_TO == "zip in the outputs folder":
    print("The files stay in", OUT.resolve(), "(on your own computer they are kept there; Colab deletes them when the session ends).")
elif SAVE_TO == "download zip (Colab)":
    if IN_COLAB:
        from google.colab import files
        files.download(str(zip_path))
    else:
        print("Not in Colab, so there is nothing to download: the zip is at", zip_path.resolve())
elif SAVE_TO == "copy to Google Drive (Colab)":
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        copy_results(DRIVE_ROOT / (FOLDER or "WetlandMapper_results"))
    else:
        print("Not in Colab. Choose 'copy to a folder' and give a folder path, or use the zip at", zip_path.resolve())
else:
    if not FOLDER:
        raise ValueError("Give a folder path in FOLDER, for example ~/wetland_results")
    copy_results(FOLDER)

Open the GeoTIFFs in QGIS: the class colours are inside the file. The wetland objects come as a GeoPackage (QGIS, ArcGIS) and a KMZ (Google Earth, a phone). `*_report.md` holds the sentences; `*_run_manifest.json` records the settings and the software version: keep it with the maps.

## If something goes wrong

| Message | What to do |
|---|---|
| "needs Earth Engine" | Tick *USE_EARTH_ENGINE* in the setup cell, enter your project ID, run setup again |
| "sample data ... not in data/cache/ yet" | The workshop leader has not downloaded them yet, or switch on Earth Engine to download them once |
| "No polygons found" | Your file holds points or lines. Draw a polygon and save again |
| "has no ... coordinate system" | A shapefile is missing its `.prj`. Upload it, or save as GeoPackage |
| "too large for this notebook" | Use a smaller outline |
| "No scene has at least 40% valid pixels" | Increase DAYS_BACK, lower the minimum, or try after the next clear satellite pass |
| The run is slow | Earth Engine fetches 36 years one by one: minutes for a small wetland, much longer for a large area |
| A year is missing | Earth Engine returned no scenes or a download failed twice. It is left out of the statistics |

## Reading results, and what this notebook does not do

- Open water is seen well. **Small, vegetated or briefly wet features** (springs, seeps, reed beds) are seen poorly: a 30 m pixel covers 0.09 ha.
- A yearly composite records the wettest cloud-free moment of the year, not the whole year. **Cover-type classes are spectral**, not a verified habitat; in tidal places the tide at the overpass matters.
- A wetland on **steep, shaded ground** can be confused with shadow, and in a **dry climate** water can be too brief for a yearly composite. WetlandMapper has options for both, which this notebook does not run: `fetch(..., dem_mask=True)` masks steep ground, and `fetch(..., climate_adaptive=True)` chooses the month to sample from rainfall. See the explore notebook.
- **Always report with the map:** the index, sensor, composite rule (percentile and months), thresholds, years and software version. The run manifest has them.

### Data and citation

- Software: Singh, M. (2026). WetlandMapper. https://doi.org/10.5281/zenodo.18967176
- Dynamics: Singh, M., and Sinha, R. (2022). *Remote Sensing Letters*, 13(1), 1-13. https://doi.org/10.1080/2150704X.2021.1980919
- Cover types: Singh, M., Allaka, S., Gupta, P. K., Patel, J. G., and Sinha, R. (2022). *Environmental Monitoring and Assessment*, 194(12), 878. https://doi.org/10.1007/s10661-022-10541-7
- Agulhas Plain Ramsar wetlands layer, supplied by SANParks (Ruth-Mary Fisher); a subset of the National Wetland Map 5. Please cite: Van Deventer, H., Smith-Adao, L., Mbona, N., Petersen, C., Skowno, A., Collins, N.B., Grenfell, M., Job, N., Lötter, M., Ollis, D., Scherman, P., Sieben, E., and Snaddon, K. (2018). South African National Biodiversity Assessment 2018: Technical Report, Volume 2a: South African Inventory of Inland Aquatic Ecosystems (SAIIAE), version 3. CSIR and SANBI, Pretoria. http://hdl.handle.net/20.500.12143/5847
- Langebaan Ramsar site boundary: Ramsar Sites Information Service (https://rsis.ramsar.org).
- Landsat Collection 2 Level-2 (USGS and NASA), Sentinel-2 surface reflectance (ESA and Copernicus), ERA5-Land (Copernicus and ECMWF), through Google Earth Engine (Gorelick et al., 2017, *Remote Sensing of Environment* 202, 18-27).